In [ ]:
#| include: false
#| echo: false
# Setup cell: makes the notebook work in Jupyter *and* in both Quarto builds.
import os, sys
from pathlib import Path

if "COURSE_LECTURE_DIR" not in os.environ:          # interactive use
    os.environ["COURSE_LECTURE_DIR"] = str(Path.cwd())
    sys.path.insert(0, str(Path.cwd().parents[1] / "scripts"))

import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
import skimage.filters as flt
import pandas as pd
import matplotlib.patches as patches
import tifffile as tiff
from lecture8_support import *
import confmap as cm
import sys
sys.path.append('../common/')
# import plotsupport as ps
import pointcloud as pc
from IPython.display import Markdown, display

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

dcolors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%matplotlib inline
%config InlineBackend.figure_format = 'retina' # This is get sharper plots on digh density screens like the Retina screen on Macs

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

# Handling uncertainties in image processing

## Why do we need to talk about the uncertain?

::: {.content-visible when-format="pdf"}

The physicist Richard Feynman nicely provided a motivation for why we need to talk about uncertainties.

:::

:::: {.columns}
::: {.column width=70%}

> “Scientific knowledge is a body of statements 
> of varying degree of certainty  
>
> – some most unsure, some nearly sure, 
> but none absolutely certain.”    
>
> @feynman1997SurelyJoking
 
:::
::: {.column width=30%}

![Richard Feyman (Credit: Caltech Archives)](figures/feynman01-NEWS-WEB.width-450_3YN6pEE.jpg){width=50%}

:::
::::

::: {.content-visible when-format="pdf"}

There is no measurement that doesn't have an uncertainty. The question how we use this information in our analysis. Determining the uncertainty is not a trivial task in image processing.

:::

## What we want

:::: {.columns}
::: {.column width=70%}   

Error bars beyond the confidence interval of the noise
    
- Each measurement has an uncertainty
- Uncertainty propagation in image processing is non-trivial

:::
::: {.column width=30%}

![xkcd -2110](figures/xkcd_errorbars.png){width=75%}

:::
::::

::: {.content-visible when-format="pdf"}

Imaging and image processing are no exception to the fact that there uncertainties related the all measurement. The problem is that there is a chain of uncertainties to consider in the uncertainty analysis. Knowing each is a non-trivial task. In the following sections, you will see some examples of uncertainty analysis.

:::

## Error or uncertainty?

:::: {.columns}
::: {.column width=50%}

### Error
The difference between measured value $c'$  
    and true value $c^*$
    
$$e = |c'-c^*| $$
    
You need a ground truth!

:::
::: {.column width=50%}
    
### Uncertainty
    
The quantification of the doubt about the measurement result. 
    
$$u_{range}= [c' - u, c' + u ]$$
    
::: 
::::

![What is error and what is uncertainty? @gillmann2019ImageUncertainty](figures/error-uncertain.png){#fig-erroruncertainty width=50%}
    
::: {.content-visible when-format="pdf"}

We often (sloppily) talk about measurement errors, but it is mostly the uncertainty we mean. 

Figure @fig-erroruncertainty illustrates how uncertainty and error are defined. The error needs the ground truth to be determined. The uncertainty is an interval around the measured value, like the confidence interval, in which the true value is included.

:::   
 



## Which uncertainties are we dealing with?

:::: {.columns}
::: {.column width=50%}

### Technical uncertainties

- Metric
- SNR
- Unsharpness
- Effects from image processing
- Numerical errors 
    
:::
::: {.column width=50%}

### Field specific uncertainties

- Model simplifications
- Variations in population
- Noise

:::
::::

## Uncertainty categories

:::: {.columns}
::: {.column width=50%}

### Systematic

Reproducible - no statistical analysis

- Rounding errors 
- Uncalibrated systems
- Algorithmic choices
- Etc.

::: 
::: {.column width=50%}

### Random

Statistical fluctuations

- Natural variations in studied population
- Source fluctuations
- Detector noise

:::
::::

## Target practicing

::: {.content-visible when-format="pdf"}

The uncertainty has two components precision and accuracy. The accuracy tells how far away our measurement is from the expected true value. The precision tell how widespread the measurements are. 

:::

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Accuracy and precision of the target practice."
#| label: fig-targetpractice
precacc(1000,figsize=(1.25*10/2.54,1.25*8/2.54),fontsize=10,dotsize=10)
plt.show()

## Target practicing in statistical terms

::: {.content-visible when-format="pdf"}

Precision and accuracy can be translated in statistical terms such that 

- The precicion corresponds to the variance of the measured values, i.e. high precision means that the variance if low. 
- The accuracy is translated to a bias in the measurement, i.e. low accuracy means that the bias is great. 

:::

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Target practice translated into bias and variance"
#| label: fig-targetbiasvariance

precacc(1000,['Large bias','Small bias','Large variance','Small variance'],figsize=(1.25*10/2.54,1.25*8/2.54),fontsize=10,dotsize=10)
plt.show()

## Counting or measuring

::: {.content-visible when-format="pdf"}

We always have uncertainties in our measurements. Some, more obvious than others. Here, we will look into the two cases counting and measuring.

:::

::::: {.columns}
:::: {.column width=50%}

### Counting    

- Discrete countable items
- Absolute values
    
Uncertainties: from preparation  

_Example:_ area of segmented region
        
![Counting items or features is one type of quantification in images.](figures/np_counting.png){#fig-counting height=150px}

::: {.content-visible when-format="pdf"}

The smallest countable item in an image is the pixel. The area is determined as the sum of the pixels in a specified region. Some perimeter metrics are based on counting the boundary pixels.

Counting seems like an absolute task, just count labelled regions and tell how many there are. The uncertainties originate from the image preparation leading to the regions:

1. Does image noise affect the number of detected items?
2. How well was the image segmented? How many misclassified pixels are there?

:::

::::
:::: {.column width=50%}

### Measuring

- Physical quantities
- Values with uncertainties

_Uncertainties:_ Noise, instrumentation  

_Example:_ mixing ratios from gray levels

![Measuring graylevels is quantifies physical relatations.](figures/np_measure.png){#fig-measure height=150px}

::: {.content-visible when-format="pdf"}

In some experiments we observe changes in the gray levels. The gray level has several sources of uncertainty

- The conversion from measured radiation to a signal that can be measured by the detector
- The quantization. How many levels were used in the acquisition.
- Noise
    - counting noise from the radiation
    - detector noise
    - quantization noise
- Algorithmic uncertainty. The reconstruction may be biased.

:::
::::
:::::

## Absolute vs relative uncertainties
Uncertainties must be compared from a neutral perspective

:::: {.columns}
::: {.column width=50%}

### Absolute uncertainty
Assume we can measure a distance with an uncertainty of 1mm

> Distance to the moon - 384400km

> The diameter of a coin - 20mm

:::
::: {.column width=50%}      
        
### Relative uncertainty
Relates the uncertainty to the measured quantity
    
> Distance to the moon - 0.000000002601457

> Diameter of a coin - 0.05
    
:::
::::

## Propagation of uncertainty

### Uncertainty of _f(x)_

$$\sigma_f(x) = \frac{\partial f}{\partial x} \cdot{} \sigma_{x}$$        
    

### Uncertainty of _f(x,y)_

$$\sigma_f(x,y)^2 = \left(\frac{\partial f}{\partial x}\cdot{} \sigma_{x}\right)^2 + \left(\frac{\partial f}{\partial y}\cdot{} \sigma_{y}\right)^2 + \underbrace{\frac{\partial f}{\partial x}\cdot{}\frac{\partial f}{\partial y}\cdot\overline{\sigma_{xy}}}_{\mbox{x and y uncorrelated?}}$$  

## Uncertain quantities in imaging experiments

- Pixel size
- Segmentation
- Sampling time stamps in time series
- Intensity levels

![Categorization of uncertainties in imaging.](figures/Uncertainties){#fig-uncertaintysources width=90%}

***
### Three examples

We want to measure:

::::: {.columns}
:::: {.column width=30%}

::: {.content-visible when-format="pdf"}

The pixel size is a fundamental value we need in most quantitative analysis in imaging. The uncertainty will therefore propagate all analyzes. Thus, it is important to measure it with high accuray and precision and naturally also to determine the uncertainty of the measurement.

:::

![Measure pixel size](figures/np_caliper_5858067_000000){#fig-pixelsize height=400px}
       
::::
:::: {.column width=30%}
        
::: {.content-visible when-format="pdf"}

The perimeter of 2D objects or the surface area requires that the edge pixels are identified. The method has an impact on the accuracy of the result.

:::

![The perimeter length](figures/np_head-circumference_6420647_000000){#fig-perimeter height=400px}

::::
:::: {.column width=30%}


::: {.content-visible when-format="pdf"}

This is frequently ocurring task in neutron imaging, using the gray level of the image to estimate the water content.

:::
        
![Water volume behind a pixel](figures/np_bucket_681234_000000){#fig-watercontent height=150px}

::::
:::::

***
### Example: measure the pixel size
To measure pixel size we need:

- An object with known length
- An image of the object

$$\mbox{pixel size}=\frac{\mbox{Object length}}{\mbox{Pixel distance between edges}}$$

__The uncertainty equation__
We assume that the two measured values are uncorrelated

$$\frac{\sigma_{a/b}}{a/b}=\sqrt{\left(\frac{\sigma_a}{a}\right)^2 + \left(\frac{\sigma_b}{b}\right)^2}$$

From this tutorial @measurementsErrorAnalysis.

***
__Our measurements__

In [ ]:
#| echo: false
#| output: true
#| fig-cap: "Distance between edges in a real object and its image."
#| label: fig-edgeobject

img1 = tiff.imread('data/edge20mm_0000.tif')
pic = plt.imread('figures/edge_object.jpg')
fig,ax=plt.subplots(1,2,figsize=(1.25*12/2.54,1.25*5/2.54))

ax[0].imshow(pic)
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].set_title('Real object')

ax[1].imshow(img1,vmin=300,vmax=30000,cmap='gray');
ax[1].set_xlabel('x [pixels]')
ax[1].set_ylabel('y [pixels]');
arrow = patches.FancyArrowPatch((55,104), (526, 73),
                                 mutation_scale=20,
                                 #ec='blue',fc='cornflowerblue',
                                 color='red',
                                 arrowstyle='<|-|>',linewidth=3
                                )
ax[1].add_patch(arrow);
#ax.annotate(text='', xy=(54,104), xytext=(527,71), arrowprops=dict(arrowstyle='<->',color='yellow',lw=4))
d=np.sqrt((528-53)**2+(104-73)**2)
ax[1].set_title('Distance between edges');

plt.show()

| Quantity  | Measurement | Uncertainty   | Unit |
|:---|---:|---:|---|
| Caliper distance  | 20.0 | 0.05 | mm |
| Pixel distance    | 464.9  | 0.32  | pixels |


[A tutorial showing the detailed analysis](https://colab.research.google.com/github/ImagingELearning/resolution/blob/main/tutorials/02_PixelSize/02_PixelSize.ipynb)

***
### Pixel size with uncertainty
We know uncertainty equation for $a/b$. 

Let's plug in the pixel meassurements:

$$\frac{\sigma_{Pixelsize}}{Pixelsize}=\sqrt{\biggl({\frac{\sigma_{pixels}}{pixels} }\biggr)^2+ \biggl({\frac{\sigma_{length}}{length}}\biggr)^2}$$

In [ ]:
#| echo: true
#| output: true
length       = 20.0 # mm
error_length = 0.05 # mm
pixels       = 464.9 # pixels
error_pixels = 0.32  # pixels

pixel_size = length/pixels

rel_uncertainty = np.sqrt((error_pixels/pixels)**2 + (error_length/length)**2)

display(Markdown('Pixel size = {0:0.2f} $\mu$m +/- {1:0.2f}$\mu$m'.format(pixel_size*1000,pixel_size*rel_uncertainty*1000)))

***
### The uncertainties package
Uncertainties can be tricky to compute. 

The [uncertainties package](https://uncertainties.readthedocs.io/en/latest/) or [PunPy](https://punpy.readthedocs.io/en/latest/) take some of this burden from you!

In [ ]:
#| echo: true
#| output: true
from uncertainties import ufloat

length = ufloat(20.0,0.05)
pixels = ufloat(464.9,0.32)

pixel_size = length/pixels

print(pixel_size)

## Measurements in segmented images

There are many methods to measure

- Area 
- Perimeter 
- Positions 
- Distances

Considering them to be absolute values as we mostly just count pixels...



... Let's see how the uncertainty of the pixel size among others changes this

***
### Meassure the perimeter length

The perimeter length has three sources of uncertainty:

- How well was the image segmented?  
    _Edges are segmented with least confidence_

- How were the edge pixels identified?  
    _Method choices can introduce biases_

- The pixel size... 


***
### Selecting edge pixels

:::: {.columns}
::: {.column width=70%}
A method to identify edge pixels is 

$$edge(f) = f - \varepsilon_{SE}(f)$$

which pixels depends on the used SE.

:::
::: {.column width=30%}

![](figures/edge_detail.png){#fig-edge-detail width=250px}

:::
::::

### Counting perimeter elements

::: {.content-visible when-format="pdf"}
The perimeter can be computed using many different metrics. Figures @fig-edge_detail_pixelcount, fig-edge_detail_outline, and fig-edge_detail_centerline shows some alternatives and also that the perimeter length depends on the metric. These are pixel and neighborhood based. We can obtain improved precision with methods that, for example, fits a function to the edge gradient.
:::

:::: {.columns}
::: {.column width=30%}

![Counting pixels (18)](figures/edge_detail_pixel_count.png){#fig-edge_detail_pixelcount width=256px}

:::
::: {.column width=30%}

![Edges facing background (24)](figures/edge_detail_outline.png){#fig-edge_detail_outline width=256px}

:::
::: {.column width=30%}

![Center line (20.5)](figures/edge_detail_centerline.png){#fig-edge_detail_centerline width=256px}

:::
::::

***
### Perimeter length with uncertainty

Let's assume we can trust the segmentation:

- Edge is 18 pixels long
- The pixel size is 43.02 $\mu$m $\pm$  0.11$\mu$m

__Perimeter uncertainty equation__

In this case we sum the edge pixels 

::: {.content-visible when-format="pdf"}
_Derivation of the equation_

The uncertainty of a sum of measurements $\sum^N_{i=1}x_i$ each with all $x_i$ uncorrelated and the same uncertainty $\sigma_x$. 

Lets start with the derivative:
$\frac{\partial f}{\partial x_i}=1\quad \forall i$

now the uncertainty is the sum of all $x_i$  

${\sigma_{\sum x_i}} = \sum_{i=1}^N {\sigma_x} = N {\sigma_x}$

which leads to 
:::

$$\sigma_{\sum x_i} = \sigma_x \cdot N$$

We have $\sigma_{Pixelsize} = 0.11\,\mu{}m$, which gives $\sigma_{Edgelength} = 0.11 \cdot 18=1.98 \mu{}m$

The edge length is 774 $\pm$ 2.0 $\mu{}m$

*** 
### Sampling vs. volume error

::: {.content-visible when-format="pdf"}
The sampling theorem tells us the smallest feature we can detect in an image. The shape of features at this limit will not be well described. @fig-sphereserror illustrates this with a sphere with increasing radius. The measured volume is compared to the theoretical volume using the equation of the sphere volume. The error plot shows that a radius of about 10 is required to reach an error below 1%. 

The spheres in this example were centered on the mid point of a voxel. This rarely happens in real measurement, which results in greater uncertainties due to the skewed voxel representation of the sphere. 

The volume error from the sampleing will increase the uncertainty in addition to the pixel size uncertainty.
:::

![The effect of object size relative to the pixel size demonstrated with spheres.](figures/spheres_error){#fig-sphereserror width=50%}

***
### Measure water volume from gray levels
In neutron imaging it is common to quantify the water content from radiographs.

:::: {.columns}
::: {.column width=70%}

- The transmission (gray level) is  
    $T=\frac{I}{I_0}=e^{-\mu \cdot d}$   
- The metric pixel area (pixel size)

    
The water volume is $V_{water}=pixelsize^2 \cdot d$
    
<div style="margin-bottom: 2em;"></div>
    
_Our uncertainties are now:_

- Standard deviation of the transmission (confidence interval)
- The pixel size ... again
    
:::
::: {.column width=30%}

![Block wise distribution of materials behind a pixel](figures/pixel_volume.png){#fig-watervolume width=400px}
    
:::
::::

## Volume with uncertainty

Let's assume we can trust the segmentation:

- Thickness $d=5\pm0.1~mm$
- The pixel size is $p=43 \pm 0.11~\mu{}m$


### Volume uncertainty equation
In this case, we know the uncertainty of $V=p^2\cdot d$

::: {.content-visible when-format="pdf"}

_Derivation of the equation_

The uncertainty of a multi product $\prod^N_{i=1}x_i$ each with all $x_i$ uncorrelated and the uncertainty $\sigma_{x_i}$, or more specific $x^2\cdot y$ with $\sigma_{x}$ $\sigma_{y}$. 

Lets start with the derivative:  

$$\frac{\partial (x^2\cdot y)}{\partial x}=2xy$$

and  

$\frac{\partial (x^2\cdot y)}{\partial y}=x^2$

Now the uncertainty is 
$${\sigma_{x^2\cdot y}}^2 = \left(2xy \cdot \sigma_x \right)^2+ \left(x^2 \cdot \sigma_y \right)^2 $$

Dividing both sides by $x^2\cdot y$ and taking the square root to get the relative uncertainty

$$\frac{{\sigma_{x^2\cdot y}}}{x^2\cdot y} = \sqrt{\frac{\left( 2xy \cdot \sigma_x \right)^2}{\left( x^2\cdot y\right)^2}+ \frac{\left( x^2 \cdot \sigma_y \right)^2}{\left(x^2\cdot y\right)^2} } = \sqrt{4\left( \frac{\sigma_x }{x}\right)^2+ \left( \frac{\sigma_y}{y}\right)^2}$$

:::

Which gives the relative uncertainty 
$$\frac{\sigma_V}{V}=\sqrt{4\left(\frac{\sigma_{Pixelsize}}{Pixelsize}\right)^2 + \left(\frac{\sigma_{d}}{d}\right)^2}$$

***
### Computing the volume with uncertainty 
Plugging in the measurements in the equation:

$$\frac{\sigma_V}{V}=\sqrt{4\left(\frac{\sigma_{Pixelsize}}{Pixelsize}\right)^2 + \left(\frac{\sigma_{d}}{d}\right)^2}$$

In [ ]:
#| echo: true
#| output: true
d           = 5.0      # mm
d_uncertain = 0.1      # mm
p           = 0.043    # mm
p_uncertain = 0.00011  # mm

V = p**2 * d

rel_uncertainty = np.sqrt(4*(p_uncertain/p)**2 + (d_uncertain/d)**2)

display(Markdown('Volume V = ({0:0.5f} +/- {1:0.5f}) mm$^3$'.format(V,rel_uncertainty*V)))